# 🧠 EX52: Model Configurations (การกำหนดค่าคอนฟิกโมเดล)

| วิธี | ไฟล์ | ค่าน้ำหนัก | เหมาะกับ |
|-----|------|-----------|---------|
| Scratch (YAML) | `yolov8n.yaml` | Kaiming random init | Domain ใหม่ทั้งหมด |
| Pretrained (.pt) | `yolov8n.pt` | COCO-pretrained | Transfer learning ✅ |
| Hybrid | `yaml` + `.load(pt)` | Transfer weights to custom | ทดลองปรับโครงสร้าง |

### Kaiming (He) Initialization
$$W \sim \mathcal{N}\left(0,\ \sqrt{\tfrac{2}{n_{in}}}\right)$$
รักษา variance ของสัญญาณทุก layer ป้องกัน vanishing/exploding gradient

## 🔗 ลิงก์
- [[YOLO_Learning_Plan]] | [[learning_journal]]


In [ ]:
import gc, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
%matplotlib inline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] ใช้อุปกรณ์: {device}")

print("\n--- เริ่มการตรวจสอบ ---")
print("1. โหลดแบบ Scratch (YAML)")
yaml_model = YOLO("yolov8n.yaml")
print("2. โหลดแบบ Pretrained (.pt)")
pretrained_model = YOLO("yolov8n.pt")
print("3. โหลดแบบ Hybrid (YAML + PT)")
hybrid_model = YOLO("yolov8n.yaml")
hybrid_model.load("yolov8n.pt")

yaml_p = sum(p.numel() for p in yaml_model.model.parameters())
pt_p   = sum(p.numel() for p in pretrained_model.model.parameters())
hyb_p  = sum(p.numel() for p in hybrid_model.model.parameters())

print(f"\nYAML params:       {yaml_p:>12,} พารามิเตอร์")
print(f"Pretrained params: {pt_p:>12,} พารามิเตอร์")
print(f"Hybrid params:     {hyb_p:>12,} พารามิเตอร์")

w = next(yaml_model.model.parameters())
print(f"\nYAML layer แรก (Random) | mean={w.mean().item():.6f} std={w.std().item():.6f}")
print(f"  Kaiming คาด std ≈ {(2/w.shape[1])**0.5:.4f}")

w_pt = next(pretrained_model.model.parameters())
print(f"Pretrained layer แรก     | mean={w_pt.mean().item():.6f} std={w_pt.std().item():.6f}")

dummy = torch.randn(1, 3, 320, 320)
yaml_model.model.eval(); pretrained_model.model.eval()
with torch.no_grad():
    out_yaml = yaml_model.model(dummy)
    out_pt   = pretrained_model.model(dummy)
print(f"\nYAML output type: {type(out_yaml).__name__}")
print("--- สิ้นสุดการตรวจสอบ ---")

del yaml_model, pretrained_model, hybrid_model, dummy, out_yaml, out_pt
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

plt.figure(figsize=(8,4))
plt.bar(["Random (YAML)", "Pretrained (.pt)", "Hybrid"], [yaml_p, pt_p, hyb_p], color=["#e74c3c","#3498db", "#2ecc71"])
plt.ylabel("Parameters"); plt.title("Random vs Pretrained vs Hybrid Parameters")
for i,v in enumerate([yaml_p, pt_p, hyb_p]):
    plt.text(i, v*1.01, f"{v/1e6:.2f}M", ha="center")
plt.tight_layout(); plt.show()
